# MST++: Multi-Stage Spectral-wise Transformer for NIR Reconstruction

**Architecture from Cai et al. (CVPRW 2022), adapted for RGB → NIR/RedEdge**

Based on the paper: "Leveraging a Vision Transformer and Adversarial Unsupervised Domain Adaptation 
to Generate the NIR Band on Out-of-Domain VHR RGB"

## Key Features:
- **S-MSA (Spectral-wise Multi-head Self-Attention)**: Channel-wise attention (C×C, not spatial N×N)
- **SST (Single-stage Spectral-wise Transformer)**: U-shaped encoder-decoder structure  
- **Multi-stage cascade**: Coarse-to-fine refinement
- **Target: ~1,619,625 parameters** (paper specification)
- **Loss: L1** (as per paper)
- **Output: 4 bands** (B5, B6, B7, B8) instead of original 1 band

## 1. Configuration and Imports

In [ ]:
import tensorflow as tf
import numpy as np
import os
import math
from tensorflow import keras
from tensorflow.keras import layers

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
class Config:
    # GCS Data
    BUCKET = 'YOUR_BUCKET'
    FOLDER = 'dataset_training_mst'
    PATCH_SIZE = 128          # Paper uses 128x128
    INPUT_CHANNELS = 3
    OUTPUT_CHANNELS = 4       # B5, B6, B7, B8 (paper uses 1)

    # Data pipeline
    NUM_PARALLEL_READS = tf.data.AUTOTUNE
    TFRECORD_COMPRESSION = 'GZIP'
    USE_LOCAL_DATA = False
    LOCAL_DATA_DIR = '/content/tfrecord_cache'
    CACHE_VAL = True
    CACHE_TEST = True
    CACHE_DIR = '/content/tfdata_cache'
    DETERMINISTIC_DATA = False

    # MST++ Architecture - PAPER VALUES (Cai et al. CVPRW 2022)
    # Original paper: ~1,619,625 params for 1 output channel
    # Our model: ~1,721,166 params for 4 output channels
    DIM = 28                  # Base embedding dimension (paper: 28)
    NUM_STAGES = 3            # Number of SST stages in cascade (paper: 3)
    NUM_BLOCKS = [2, 2, 2]    # SAB blocks per encoder level (paper: [2,2,2])
    NUM_HEADS = [1, 2, 4]     # Attention heads per level (paper: [1,2,4])
    FFN_EXPANSION = 2.66      # FFN expansion factor (paper: 2.66)

    # Training - Adjusted for stability
    BATCH_SIZE = 8            # Paper uses 8
    EPOCHS = 300              # Paper uses 300
    LEARNING_RATE = 1e-4      # Reduced from 4e-4 due to sigmoid output
    WEIGHT_DECAY = 0.0

    # Warmup and LR schedule
    WARMUP_EPOCHS = 5
    LR_MIN = 1e-6

    # Gradient clipping for stability
    GRADIENT_CLIP_NORM = 1.0
    
    # Adam epsilon for numerical stability
    ADAM_EPSILON = 1e-7

    # Drive checkpointing
    SAVE_TO_DRIVE = True
    DRIVE_PATH = '/content/drive/MyDrive/MSTPP_Checkpoints'

    EST_EXAMPLES_PER_FILE = 495

config = Config()

print("MST++ Configuration (Paper values)")
print(f"  Input: {config.PATCH_SIZE}x{config.PATCH_SIZE}x{config.INPUT_CHANNELS}")
print(f"  Output: {config.PATCH_SIZE}x{config.PATCH_SIZE}x{config.OUTPUT_CHANNELS}")
print(f"  Architecture:")
print(f"    - DIM: {config.DIM}")
print(f"    - NUM_STAGES: {config.NUM_STAGES}")
print(f"    - NUM_BLOCKS: {config.NUM_BLOCKS}")
print(f"    - NUM_HEADS: {config.NUM_HEADS}")
print(f"  Training:")
print(f"    - BATCH_SIZE: {config.BATCH_SIZE}")
print(f"    - LR: {config.LEARNING_RATE} (warmup {config.WARMUP_EPOCHS} epochs)")
print(f"    - EPOCHS: {config.EPOCHS}")

## 2. MST++ Building Blocks

### Architecture Overview:
```
Input (H, W, 3) 
    → Conv3x3 → (H, W, DIM)
    → SST Stage 1 → SST Stage 2 → SST Stage 3  [Cascade]
    → Conv3x3 → (H, W, 4)
Output (H, W, 4)
```

### SST (Single-stage Spectral-wise Transformer) - U-shaped:
```
Encoder: SAB → Downsample → SAB → Downsample → SAB
Bottleneck: SAB
Decoder: Upsample + Skip → SAB → Upsample + Skip → SAB → SAB
```

### SAB (Spectral-wise Attention Block):
```
x → LayerNorm → S-MSA → + → LayerNorm → FFN → + → output
         ↑_____________↑          ↑________↑
```

### S-MSA (Spectral-wise Multi-head Self-Attention):
- Treats each **channel** as a token (not spatial positions)
- Computes attention between C channels: Q, K, V ∈ (B, C, H*W)
- Captures spectral correlations for RGB→NIR translation

In [ ]:
# ============================================================================
# MST++ BUILDING BLOCKS
# ============================================================================

class SMSA(layers.Layer):
    """
    Spectral-wise Multi-head Self-Attention (S-MSA).
    
    Key insight: Instead of spatial attention (N×N where N=H*W),
    we compute channel attention (C×C) by treating each channel as a token.
    
    This captures spectral correlations crucial for RGB→NIR translation.
    """
    def __init__(self, dim, num_heads=1, bias=False, **kwargs):
        super().__init__(**kwargs)
        self.dim = dim
        self.num_heads = num_heads
        self.bias = bias
        self.head_dim = dim // num_heads

    def build(self, input_shape):
        # Q, K, V projections (1x1 conv)
        self.qkv = layers.Conv2D(
            self.dim * 3, 1, use_bias=self.bias,
            kernel_initializer='glorot_uniform',
            name='qkv'
        )
        # Output projection
        self.proj = layers.Conv2D(
            self.dim, 1, use_bias=self.bias,
            kernel_initializer='glorot_uniform',
            name='proj'
        )
        super().build(input_shape)

    def call(self, x, training=None):
        # x: (B, H, W, C)
        shape = tf.shape(x)
        B, H, W, C = shape[0], shape[1], shape[2], shape[3]
        
        # Compute Q, K, V
        qkv = self.qkv(x)  # (B, H, W, 3*C)
        qkv = tf.reshape(qkv, [B, H * W, 3, self.num_heads, self.head_dim])
        qkv = tf.transpose(qkv, [2, 0, 3, 4, 1])  # (3, B, heads, head_dim, H*W)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        # Scaled dot-product attention
        # Scale by 1/seq_len to prevent large values (seq_len=16384 for 128x128)
        seq_len = tf.cast(H * W, tf.float32)
        scale = 1.0 / seq_len
        
        # Spectral attention: (B, heads, head_dim, head_dim) 
        attn = tf.matmul(q, k, transpose_b=True) * scale
        
        # Clip before softmax to prevent overflow
        attn = tf.clip_by_value(attn, -30.0, 30.0)
        attn = tf.nn.softmax(attn, axis=-1)
        
        # Apply attention to values
        out = tf.matmul(attn, v)  # (B, heads, head_dim, H*W)
        
        # Reshape back
        out = tf.transpose(out, [0, 3, 1, 2])  # (B, H*W, heads, head_dim)
        out = tf.reshape(out, [B, H, W, C])
        
        # Output projection
        out = self.proj(out)
        return out


class GDFN(layers.Layer):
    """
    Gated-Dconv Feed-Forward Network.
    
    Uses depthwise convolution for local feature mixing
    and a gating mechanism for feature selection.
    """
    def __init__(self, dim, ffn_expansion=2.66, bias=False, **kwargs):
        super().__init__(**kwargs)
        self.dim = dim
        self.hidden_dim = int(dim * ffn_expansion)
        self.bias = bias

    def build(self, input_shape):
        self.project_in = layers.Conv2D(
            self.hidden_dim * 2, 1, use_bias=self.bias,
            kernel_initializer='glorot_uniform',
            name='project_in'
        )
        self.dwconv = layers.DepthwiseConv2D(
            3, padding='same', use_bias=self.bias,
            depthwise_initializer='glorot_uniform',
            name='dwconv'
        )
        self.project_out = layers.Conv2D(
            self.dim, 1, use_bias=self.bias,
            kernel_initializer='glorot_uniform',
            name='project_out'
        )
        super().build(input_shape)

    def call(self, x, training=None):
        x = self.project_in(x)
        x1, x2 = tf.split(x, 2, axis=-1)
        # Clip inputs to prevent extreme values
        x1 = tf.clip_by_value(x1, -10.0, 10.0)
        x2 = tf.clip_by_value(x2, -10.0, 10.0)
        x = self.dwconv(x1) * tf.nn.gelu(x2)
        # Clip output
        x = tf.clip_by_value(x, -10.0, 10.0)
        x = self.project_out(x)
        return x


class SAB(layers.Layer):
    """
    Spectral-wise Attention Block (SAB).
    
    Structure: LayerNorm → S-MSA → + → LayerNorm → GDFN → +
    """
    def __init__(self, dim, num_heads=1, ffn_expansion=2.66, bias=False, **kwargs):
        super().__init__(**kwargs)
        self.dim = dim
        self.num_heads = num_heads
        self.ffn_expansion = ffn_expansion
        self.bias = bias

    def build(self, input_shape):
        self.norm1 = layers.LayerNormalization(epsilon=1e-5, name='norm1')
        self.attn = SMSA(self.dim, self.num_heads, self.bias, name='attn')
        self.norm2 = layers.LayerNormalization(epsilon=1e-5, name='norm2')
        self.ffn = GDFN(self.dim, self.ffn_expansion, self.bias, name='ffn')
        super().build(input_shape)

    def call(self, x, training=None):
        # Scale residuals by 0.1 to prevent gradient explosion in deep networks
        x = x + 0.1 * self.attn(self.norm1(x), training=training)
        x = x + 0.1 * self.ffn(self.norm2(x), training=training)
        return x


class Downsample(layers.Layer):
    """Pixel unshuffle for downsampling."""
    def __init__(self, dim, **kwargs):
        super().__init__(**kwargs)
        self.dim = dim

    def build(self, input_shape):
        self.conv = layers.Conv2D(
            self.dim * 2, 1, use_bias=False,
            kernel_initializer='glorot_uniform',
            name='conv'
        )
        super().build(input_shape)

    def call(self, x, training=None):
        shape = tf.shape(x)
        B, H, W, C = shape[0], shape[1], shape[2], shape[3]
        x = tf.reshape(x, [B, H // 2, 2, W // 2, 2, C])
        x = tf.transpose(x, [0, 1, 3, 2, 4, 5])
        x = tf.reshape(x, [B, H // 2, W // 2, C * 4])
        x = self.conv(x)
        return x


class Upsample(layers.Layer):
    """Pixel shuffle for upsampling."""
    def __init__(self, dim, **kwargs):
        super().__init__(**kwargs)
        self.dim = dim

    def build(self, input_shape):
        self.conv = layers.Conv2D(
            self.dim * 2, 1, use_bias=False,
            kernel_initializer='glorot_uniform',
            name='conv'
        )
        super().build(input_shape)

    def call(self, x, training=None):
        x = self.conv(x)
        x = tf.nn.depth_to_space(x, 2)
        return x


class SST(layers.Layer):
    """
    Single-stage Spectral-wise Transformer (U-shaped).
    
    Encoder-decoder structure with skip connections.
    3 levels: [H, W] → [H/2, W/2] → [H/4, W/4]
    """
    def __init__(self, dim, num_blocks, num_heads, ffn_expansion=2.66, **kwargs):
        super().__init__(**kwargs)
        self.dim = dim
        self.num_blocks = num_blocks
        self.num_heads = num_heads
        self.ffn_expansion = ffn_expansion

    def build(self, input_shape):
        # Encoder level 0
        self.enc0 = [
            SAB(self.dim, self.num_heads[0], self.ffn_expansion, name=f'enc0_sab{i}')
            for i in range(self.num_blocks[0])
        ]
        self.down0 = Downsample(self.dim, name='down0')
        
        # Encoder level 1
        self.enc1 = [
            SAB(self.dim * 2, self.num_heads[1], self.ffn_expansion, name=f'enc1_sab{i}')
            for i in range(self.num_blocks[1])
        ]
        self.down1 = Downsample(self.dim * 2, name='down1')
        
        # Bottleneck
        self.bottleneck = [
            SAB(self.dim * 4, self.num_heads[2], self.ffn_expansion, name=f'bottleneck_sab{i}')
            for i in range(self.num_blocks[2])
        ]
        
        # Decoder level 1
        self.up1 = Upsample(self.dim * 4, name='up1')
        self.reduce1 = layers.Conv2D(self.dim * 2, 1, use_bias=False, name='reduce1')
        self.dec1 = [
            SAB(self.dim * 2, self.num_heads[1], self.ffn_expansion, name=f'dec1_sab{i}')
            for i in range(self.num_blocks[1])
        ]
        
        # Decoder level 0
        self.up0 = Upsample(self.dim * 2, name='up0')
        self.reduce0 = layers.Conv2D(self.dim, 1, use_bias=False, name='reduce0')
        self.dec0 = [
            SAB(self.dim, self.num_heads[0], self.ffn_expansion, name=f'dec0_sab{i}')
            for i in range(self.num_blocks[0])
        ]
        
        super().build(input_shape)

    def call(self, x, training=None):
        # Encoder
        enc0_out = x
        for blk in self.enc0:
            enc0_out = blk(enc0_out, training=training)
        
        enc1_in = self.down0(enc0_out, training=training)
        enc1_out = enc1_in
        for blk in self.enc1:
            enc1_out = blk(enc1_out, training=training)
        
        # Bottleneck
        bottleneck_in = self.down1(enc1_out, training=training)
        bottleneck_out = bottleneck_in
        for blk in self.bottleneck:
            bottleneck_out = blk(bottleneck_out, training=training)
        
        # Decoder with skip connections
        dec1_in = self.up1(bottleneck_out, training=training)
        dec1_in = self.reduce1(tf.concat([dec1_in, enc1_out], axis=-1))
        dec1_out = dec1_in
        for blk in self.dec1:
            dec1_out = blk(dec1_out, training=training)
        
        dec0_in = self.up0(dec1_out, training=training)
        dec0_in = self.reduce0(tf.concat([dec0_in, enc0_out], axis=-1))
        dec0_out = dec0_in
        for blk in self.dec0:
            dec0_out = blk(dec0_out, training=training)
        
        return dec0_out


class MSTPP(layers.Layer):
    """
    MST++ (Multi-Stage Spectral-wise Transformer).
    
    Cascades multiple SST stages for coarse-to-fine refinement.
    """
    def __init__(self, config, **kwargs):
        super().__init__(**kwargs)
        self.config = config
        self.dim = config.DIM
        self.num_stages = config.NUM_STAGES
        self.num_blocks = config.NUM_BLOCKS
        self.num_heads = config.NUM_HEADS
        self.ffn_expansion = config.FFN_EXPANSION

    def build(self, input_shape):
        # Input projection
        self.input_proj = layers.Conv2D(
            self.dim, 3, padding='same', use_bias=False,
            kernel_initializer='glorot_uniform',
            name='input_proj'
        )
        
        # Cascade of SST stages
        self.stages = [
            SST(self.dim, self.num_blocks, self.num_heads, 
                self.ffn_expansion, name=f'sst_stage{i}')
            for i in range(self.num_stages)
        ]
        
        # Output projection with small initialization
        self.output_proj = layers.Conv2D(
            self.config.OUTPUT_CHANNELS, 3, padding='same', use_bias=True,
            kernel_initializer=tf.initializers.TruncatedNormal(stddev=0.02),
            bias_initializer='zeros',
            name='output_proj'
        )
        
        super().build(input_shape)

    def call(self, x, training=None):
        x = self.input_proj(x)
        
        for stage in self.stages:
            x = stage(x, training=training) + x  # Residual between stages
        
        x = self.output_proj(x)
        # Sigmoid to bound output to [0, 1]
        x = tf.nn.sigmoid(x)
        return x


def build_mstpp(config):
    """Build MST++ model."""
    inputs = keras.Input(shape=(config.PATCH_SIZE, config.PATCH_SIZE, config.INPUT_CHANNELS))
    outputs = MSTPP(config, name='mstpp')(inputs)
    model = keras.Model(inputs, outputs, name='MSTPP')
    return model


print("MST++ architecture defined:")
print("  - SMSA: Spectral-wise Multi-head Self-Attention")
print("  - GDFN: Gated-Dconv Feed-Forward Network")
print("  - SAB: Spectral-wise Attention Block (0.1x residual scaling)")
print("  - SST: Single-stage Spectral-wise Transformer (U-shaped)")
print("  - MSTPP: Multi-stage cascade")

## 3. Loss and Metrics

**Paper uses L1 Loss** for NIR reconstruction.

In [ ]:
def l1_loss_masked(y_true, y_pred, mask, eps=1e-8):
    """L1 Loss (MAE) with validity mask - Paper's choice."""
    diff = tf.abs(y_true - y_pred)
    diff = diff * mask[..., tf.newaxis]
    num_valid = tf.reduce_sum(mask) * tf.cast(tf.shape(y_true)[-1], tf.float32) + eps
    return tf.reduce_sum(diff) / num_valid


def mrae_metric_masked(y_true, y_pred, mask, eps=1e-6):
    """MRAE for monitoring (not training)."""
    diff = tf.abs(y_true - y_pred)
    denom = tf.maximum(tf.abs(y_true), eps)
    rel = diff / denom
    rel = rel * mask[..., tf.newaxis]
    num_valid = tf.reduce_sum(mask) * tf.cast(tf.shape(y_true)[-1], tf.float32) + 1e-8
    return tf.reduce_sum(rel) / num_valid


def rmse_metric_masked(y_true, y_pred, mask, eps=1e-8):
    """RMSE with validity mask."""
    diff = (y_true - y_pred) * mask[..., tf.newaxis]
    mse = tf.reduce_sum(tf.square(diff)) / (tf.reduce_sum(mask) * tf.cast(tf.shape(y_true)[-1], tf.float32) + eps)
    return tf.sqrt(mse)


def sam_metric_masked(y_true, y_pred, mask, eps=1e-7):
    """SAM with validity mask."""
    norm_true = tf.sqrt(tf.reduce_sum(tf.square(y_true), axis=-1) + eps)
    norm_pred = tf.sqrt(tf.reduce_sum(tf.square(y_pred), axis=-1) + eps)
    dot = tf.reduce_sum(y_true * y_pred, axis=-1) / (norm_true * norm_pred + eps)
    dot = tf.clip_by_value(dot, -1.0 + eps, 1.0 - eps)
    angle = tf.acos(dot)
    angle = angle * mask
    return tf.reduce_sum(angle) / (tf.reduce_sum(mask) + eps)


@tf.function
def mstpp_loss(y_true, y_pred, mask):
    """Primary loss function: L1 (as per paper)."""
    return l1_loss_masked(y_true, y_pred, mask)

print("Loss: L1 (paper) + monitoring metrics (MRAE, RMSE, SAM)")

## 4. Data Loading

In [ ]:
def _apply_dataset_options(ds, config):
    options = tf.data.Options()
    deterministic = getattr(config, 'DETERMINISTIC_DATA', False)
    if hasattr(options, 'deterministic'):
        options.deterministic = deterministic
    if hasattr(options, 'experimental_deterministic'):
        options.experimental_deterministic = deterministic
    return ds.with_options(options)


def _maybe_cache(ds, config, name):
    flag = getattr(config, f"CACHE_{name.upper()}", False)
    if not flag:
        return ds
    cache_dir = getattr(config, 'CACHE_DIR', '/content/tfdata_cache')
    os.makedirs(cache_dir, exist_ok=True)
    return ds.cache(os.path.join(cache_dir, name))


def _maybe_copy_to_local(files, config, tag="data"):
    if not getattr(config, 'USE_LOCAL_DATA', False):
        return files
    local_dir = getattr(config, 'LOCAL_DATA_DIR', '/content/tfrecord_cache')
    os.makedirs(local_dir, exist_ok=True)
    local_files = []
    for f in files:
        dst = os.path.join(local_dir, os.path.basename(f))
        if not tf.io.gfile.exists(dst):
            try:
                tf.io.gfile.copy(f, dst)
            except Exception as e:
                print(f"Copy failed ({tag}): {e}")
                return files
        local_files.append(dst)
    print(f"Local cache: {len(local_files)} files ({tag})")
    return local_files


def _tfrecord_dataset(files, config):
    compression = getattr(config, 'TFRECORD_COMPRESSION', 'GZIP')
    if compression in ('', 'None'):
        compression = None
    return tf.data.TFRecordDataset(
        files,
        compression_type=compression,
        num_parallel_reads=getattr(config, 'NUM_PARALLEL_READS', tf.data.AUTOTUNE)
    )


def parse_tfrecord(example, config):
    """Parse TFRecord exported from GEE with validity mask."""
    feature_description = {
        'B2': tf.io.VarLenFeature(tf.float32),
        'B3': tf.io.VarLenFeature(tf.float32),
        'B4': tf.io.VarLenFeature(tf.float32),
        'B5': tf.io.VarLenFeature(tf.float32),
        'B6': tf.io.VarLenFeature(tf.float32),
        'B7': tf.io.VarLenFeature(tf.float32),
        'B8': tf.io.VarLenFeature(tf.float32),
        'valid': tf.io.VarLenFeature(tf.float32),
    }

    parsed = tf.io.parse_single_example(example, feature_description)

    def find_dimensions(num_values):
        num_values_f = tf.cast(num_values, tf.float32)
        sqrt_n = tf.sqrt(num_values_f)
        sqrt_n_int = tf.cast(sqrt_n, tf.int32)
        is_square = tf.equal(sqrt_n_int * sqrt_n_int, num_values)

        def get_square_dims():
            return sqrt_n_int, sqrt_n_int

        def find_divisor():
            candidate_heights = tf.constant([64, 65, 66, 128, 129, 130, 256], dtype=tf.int32)
            remainders = num_values % candidate_heights
            valid_mask = tf.equal(remainders, 0)
            first_valid_idx = tf.argmax(tf.cast(valid_mask, tf.int32))
            height = candidate_heights[first_valid_idx]
            width = num_values // height
            any_valid = tf.reduce_any(valid_mask)
            height = tf.cond(any_valid, lambda: height, lambda: sqrt_n_int + 1)
            width = tf.cond(any_valid, lambda: width, lambda: num_values // (sqrt_n_int + 1))
            return height, width

        return tf.cond(is_square, get_square_dims, find_divisor)

    def extract_band(band_name):
        dense = tf.sparse.to_dense(parsed[band_name])
        # Replace NaN AND Inf with zeros
        dense = tf.where(tf.math.is_nan(dense) | tf.math.is_inf(dense), tf.zeros_like(dense), dense)
        num_values = tf.shape(dense)[0]
        height, width = find_dimensions(num_values)
        return tf.reshape(dense, [height, width])

    b2, b3, b4 = extract_band('B2'), extract_band('B3'), extract_band('B4')
    rgb = tf.stack([b4, b3, b2], axis=-1)

    b5, b6, b7, b8 = extract_band('B5'), extract_band('B6'), extract_band('B7'), extract_band('B8')
    nir_rededge = tf.stack([b5, b6, b7, b8], axis=-1)

    valid_mask = extract_band('valid')

    rgb = tf.clip_by_value(rgb, 0.0, 10000.0) / 10000.0
    nir_rededge = tf.clip_by_value(nir_rededge, 0.0, 10000.0) / 10000.0

    rgb_sum = tf.reduce_sum(rgb, axis=-1)
    nir_sum = tf.reduce_sum(nir_rededge, axis=-1)
    non_black = tf.cast((rgb_sum > 0.01) | (nir_sum > 0.01), tf.float32)
    mask = valid_mask * non_black

    return tf.cast(rgb, tf.float32), tf.cast(nir_rededge, tf.float32), tf.cast(mask, tf.float32)


def random_crop_pair_with_mask(rgb, nir, mask, crop_size):
    h = tf.shape(rgb)[0]
    w = tf.shape(rgb)[1]
    max_offset_h = tf.maximum(h - crop_size, 0)
    max_offset_w = tf.maximum(w - crop_size, 0)
    offset_h = tf.random.uniform((), 0, max_offset_h + 1, dtype=tf.int32)
    offset_w = tf.random.uniform((), 0, max_offset_w + 1, dtype=tf.int32)
    rgb = rgb[offset_h:offset_h + crop_size, offset_w:offset_w + crop_size, :]
    nir = nir[offset_h:offset_h + crop_size, offset_w:offset_w + crop_size, :]
    mask = mask[offset_h:offset_h + crop_size, offset_w:offset_w + crop_size]
    rgb = tf.image.resize_with_crop_or_pad(rgb, crop_size, crop_size)
    nir = tf.image.resize_with_crop_or_pad(nir, crop_size, crop_size)
    mask = tf.image.resize_with_crop_or_pad(mask[..., tf.newaxis], crop_size, crop_size)[..., 0]
    return rgb, nir, mask


def center_crop_pair_with_mask(rgb, nir, mask, crop_size):
    h = tf.shape(rgb)[0]
    w = tf.shape(rgb)[1]
    offset_h = tf.maximum((h - crop_size) // 2, 0)
    offset_w = tf.maximum((w - crop_size) // 2, 0)
    rgb = rgb[offset_h:offset_h + crop_size, offset_w:offset_w + crop_size, :]
    nir = nir[offset_h:offset_h + crop_size, offset_w:offset_w + crop_size, :]
    mask = mask[offset_h:offset_h + crop_size, offset_w:offset_w + crop_size]
    rgb = tf.image.resize_with_crop_or_pad(rgb, crop_size, crop_size)
    nir = tf.image.resize_with_crop_or_pad(nir, crop_size, crop_size)
    mask = tf.image.resize_with_crop_or_pad(mask[..., tf.newaxis], crop_size, crop_size)[..., 0]
    return rgb, nir, mask


def augment_with_mask(rgb, nir, mask, config):
    rgb, nir, mask = random_crop_pair_with_mask(rgb, nir, mask, config.PATCH_SIZE)
    if tf.random.uniform(()) > 0.5:
        rgb = tf.image.flip_left_right(rgb)
        nir = tf.image.flip_left_right(nir)
        mask = tf.image.flip_left_right(mask[..., tf.newaxis])[..., 0]
    if tf.random.uniform(()) > 0.5:
        rgb = tf.image.flip_up_down(rgb)
        nir = tf.image.flip_up_down(nir)
        mask = tf.image.flip_up_down(mask[..., tf.newaxis])[..., 0]
    k = tf.random.uniform((), 0, 4, dtype=tf.int32)
    rgb = tf.image.rot90(rgb, k)
    nir = tf.image.rot90(nir, k)
    mask = tf.image.rot90(mask[..., tf.newaxis], k)[..., 0]
    return rgb, nir, mask


def prepare_validation_with_mask(rgb, nir, mask, config):
    return center_crop_pair_with_mask(rgb, nir, mask, config.PATCH_SIZE)


def assert_finite(rgb, nir, mask):
    """Assert all tensors are finite (no NaN/Inf) - for debugging."""
    tf.debugging.assert_all_finite(rgb, "RGB has NaN/Inf")
    tf.debugging.assert_all_finite(nir, "NIR has NaN/Inf")
    tf.debugging.assert_all_finite(mask, "MASK has NaN/Inf")
    return rgb, nir, mask


def load_dataset(pattern, config, is_training=True, tag="data"):
    files = tf.io.gfile.glob(pattern)
    print(f"Found {len(files)} files")
    files = _maybe_copy_to_local(files, config, tag=tag)
    ds = _tfrecord_dataset(files, config)
    ds = ds.map(lambda x: parse_tfrecord(x, config), num_parallel_calls=tf.data.AUTOTUNE)
    if is_training:
        ds = ds.shuffle(1000)
        ds = ds.map(lambda rgb, nir, mask: augment_with_mask(rgb, nir, mask, config),
                    num_parallel_calls=tf.data.AUTOTUNE)
    else:
        ds = ds.map(lambda rgb, nir, mask: prepare_validation_with_mask(rgb, nir, mask, config),
                    num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(config.BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    ds = _maybe_cache(ds, config, name=tag)
    return _apply_dataset_options(ds, config)


def load_train_val_split(train_pattern, config, val_split=0.1):
    files = tf.io.gfile.glob(train_pattern)
    print(f"Found {len(files)} training files")
    np.random.seed(42)
    np.random.shuffle(files)
    n_val = max(1, int(len(files) * val_split))
    val_files = files[:n_val]
    train_files = files[n_val:]
    print(f"Split: {len(train_files)} train, {len(val_files)} val")

    train_files = _maybe_copy_to_local(train_files, config, tag="train")
    val_files = _maybe_copy_to_local(val_files, config, tag="val")

    # Train dataset
    train_ds = _tfrecord_dataset(train_files, config)
    train_ds = train_ds.map(lambda x: parse_tfrecord(x, config), num_parallel_calls=tf.data.AUTOTUNE)
    train_ds = train_ds.shuffle(1000)
    train_ds = train_ds.map(lambda rgb, nir, mask: augment_with_mask(rgb, nir, mask, config),
                            num_parallel_calls=tf.data.AUTOTUNE)
    # DEBUG: Assert all data is finite
    train_ds = train_ds.map(assert_finite, num_parallel_calls=tf.data.AUTOTUNE)
    train_ds = train_ds.batch(config.BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    train_ds = _apply_dataset_options(train_ds, config)

    # Val dataset
    val_ds = _tfrecord_dataset(val_files, config)
    val_ds = val_ds.map(lambda x: parse_tfrecord(x, config), num_parallel_calls=tf.data.AUTOTUNE)
    val_ds = val_ds.map(lambda rgb, nir, mask: prepare_validation_with_mask(rgb, nir, mask, config),
                        num_parallel_calls=tf.data.AUTOTUNE)
    val_ds = val_ds.batch(config.BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    val_ds = _maybe_cache(val_ds, config, name="val")
    val_ds = _apply_dataset_options(val_ds, config)

    return train_ds, val_ds

print("Data pipeline ready")

## 5. MST++ Trainer

Features:
- **Warmup + Cosine Decay** learning rate schedule
- **Gradient clipping** for stability
- **Drive checkpointing** with state restoration

In [ ]:
import json


class WarmupCosineDecay(tf.keras.optimizers.schedules.LearningRateSchedule):
    """Warmup + Cosine Decay schedule."""
    def __init__(self, initial_lr, warmup_steps, total_steps, min_lr=1e-6):
        super().__init__()
        self.initial_lr = initial_lr
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.min_lr = min_lr

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warmup_steps = tf.cast(self.warmup_steps, tf.float32)
        total_steps = tf.cast(self.total_steps, tf.float32)

        warmup_progress = step / tf.maximum(warmup_steps, 1.0)
        warmup_lr = self.min_lr + (self.initial_lr - self.min_lr) * warmup_progress

        decay_steps = total_steps - warmup_steps
        cosine_progress = (step - warmup_steps) / tf.maximum(decay_steps, 1.0)
        cosine_decay = 0.5 * (1 + tf.cos(math.pi * tf.minimum(cosine_progress, 1.0)))
        decay_lr = self.min_lr + (self.initial_lr - self.min_lr) * cosine_decay

        return tf.where(step < warmup_steps, warmup_lr, decay_lr)

    def get_config(self):
        return {
            'initial_lr': self.initial_lr,
            'warmup_steps': self.warmup_steps,
            'total_steps': self.total_steps,
            'min_lr': self.min_lr
        }


class MSTPPTrainer:
    """MST++ Trainer with warmup, gradient clipping, and Drive checkpointing."""
    
    def __init__(self, config, checkpoint_dir='checkpoints', steps_per_epoch=None):
        self.config = config
        self.checkpoint_dir = checkpoint_dir
        os.makedirs(checkpoint_dir, exist_ok=True)

        # Build model
        self.model = build_mstpp(config)
        param_count = self.model.count_params()
        print(f"MST++ params: {param_count:,}")
        print(f"  Target: ~1,619,625 (paper spec for 1 output channel)")
        
        # Learning rate schedule
        self.steps_per_epoch = steps_per_epoch or 100
        total_steps = self.steps_per_epoch * config.EPOCHS
        warmup_steps = self.steps_per_epoch * config.WARMUP_EPOCHS
        
        self.lr_schedule = WarmupCosineDecay(
            initial_lr=config.LEARNING_RATE,
            warmup_steps=warmup_steps,
            total_steps=total_steps,
            min_lr=config.LR_MIN
        )
        
        # Optimizer with gradient clipping and epsilon for stability
        self.optimizer = tf.keras.optimizers.Adam(
            learning_rate=self.lr_schedule,
            clipnorm=config.GRADIENT_CLIP_NORM,
            epsilon=getattr(config, 'ADAM_EPSILON', 1e-7)
        )

        # Checkpointing
        self.ckpt = tf.train.Checkpoint(model=self.model, optimizer=self.optimizer)
        self.ckpt_mgr = tf.train.CheckpointManager(self.ckpt, self.checkpoint_dir, max_to_keep=3)
        self.start_epoch = 0
        self.global_step = 0

        # Metrics
        self.train_l1 = tf.keras.metrics.Mean()
        self.val_l1 = tf.keras.metrics.Mean()
        self.val_mrae = tf.keras.metrics.Mean()
        self.val_sam = tf.keras.metrics.Mean()
        self.best_val = float('inf')

    def _drive_path(self):
        return getattr(self.config, 'DRIVE_PATH', '/content/drive/MyDrive/MSTPP_Checkpoints')

    def sync_to_drive(self):
        if not os.path.exists('/content/drive'):
            return
        drive_path = self._drive_path()
        os.makedirs(drive_path, exist_ok=True)
        for fname in os.listdir(self.checkpoint_dir):
            src = os.path.join(self.checkpoint_dir, fname)
            dst = os.path.join(drive_path, fname)
            if os.path.isfile(src):
                tf.io.gfile.copy(src, dst, overwrite=True)

    def restore_from_drive(self):
        if not os.path.exists('/content/drive'):
            return
        drive_path = self._drive_path()
        if not os.path.exists(drive_path):
            return
        os.makedirs(self.checkpoint_dir, exist_ok=True)
        for fname in os.listdir(drive_path):
            src = os.path.join(drive_path, fname)
            dst = os.path.join(self.checkpoint_dir, fname)
            if os.path.isfile(src):
                tf.io.gfile.copy(src, dst, overwrite=True)

    def save_checkpoint(self, epoch):
        self.ckpt_mgr.save()
        state = {
            'epoch': epoch,
            'global_step': int(self.optimizer.iterations.numpy()),
            'best_val': float(self.best_val),
            'steps_per_epoch': self.steps_per_epoch,
        }
        with open(os.path.join(self.checkpoint_dir, 'trainer_state.json'), 'w') as f:
            json.dump(state, f)
        if getattr(self.config, 'SAVE_TO_DRIVE', False):
            self.sync_to_drive()

    def load_checkpoint(self, from_drive=False):
        if from_drive:
            self.restore_from_drive()
        if self.ckpt_mgr.latest_checkpoint:
            self.ckpt.restore(self.ckpt_mgr.latest_checkpoint).expect_partial()
            print(f"Restored from {self.ckpt_mgr.latest_checkpoint}")
        state_path = os.path.join(self.checkpoint_dir, 'trainer_state.json')
        if os.path.exists(state_path):
            with open(state_path, 'r') as f:
                state = json.load(f)
            self.start_epoch = int(state.get('epoch', 0))
            self.global_step = int(state.get('global_step', 0))
            self.best_val = float(state.get('best_val', float('inf')))
            print(f"Resuming from epoch {self.start_epoch}, step {self.global_step}")

    @tf.function
    def train_step(self, rgb, nir, mask):
        # DEBUG: Assert inputs are finite
        tf.debugging.assert_all_finite(rgb, "rgb input NaN/Inf")
        tf.debugging.assert_all_finite(nir, "nir input NaN/Inf")
        tf.debugging.assert_all_finite(mask, "mask input NaN/Inf")
        
        with tf.GradientTape() as tape:
            pred = self.model(rgb, training=True)
            # DEBUG: Assert model output is finite
            tf.debugging.assert_all_finite(pred, "pred NaN/Inf (model output)")
            loss = mstpp_loss(nir, pred, mask)
            tf.debugging.assert_all_finite(loss, "loss NaN/Inf")
        
        grads = tape.gradient(loss, self.model.trainable_variables)
        # Use global norm clipping (more standard than per-variable)
        grads, global_norm = tf.clip_by_global_norm(
            [g if g is not None else tf.zeros_like(v) for g, v in zip(grads, self.model.trainable_variables)], 
            1.0
        )
        self.optimizer.apply_gradients(zip(grads, self.model.trainable_variables))
        self.train_l1.update_state(loss)
        return loss, global_norm

    @tf.function
    def val_step(self, rgb, nir, mask):
        pred = self.model(rgb, training=False)
        l1 = l1_loss_masked(nir, pred, mask)
        mrae = mrae_metric_masked(nir, pred, mask)
        sam = sam_metric_masked(nir, pred, mask)
        self.val_l1.update_state(l1)
        self.val_mrae.update_state(mrae)
        self.val_sam.update_state(sam)

    def train(self, train_ds, val_ds, resume=True, from_drive=False):
        if resume or from_drive:
            self.load_checkpoint(from_drive=from_drive)

        for epoch in range(self.start_epoch, self.config.EPOCHS):
            self.train_l1.reset_state()
            self.val_l1.reset_state()
            self.val_mrae.reset_state()
            self.val_sam.reset_state()

            # Training
            nan_detected = False
            for step, (rgb, nir, mask) in enumerate(train_ds):
                # Check for NaN in input data and mask valid fraction
                if step == 0 and epoch == 0:
                    valid_frac = tf.reduce_mean(tf.cast(mask > 0, tf.float32))
                    print(f"Data check - RGB: [{tf.reduce_min(rgb):.3f}, {tf.reduce_max(rgb):.3f}], "
                          f"NIR: [{tf.reduce_min(nir):.3f}, {tf.reduce_max(nir):.3f}], "
                          f"Valid fraction: {valid_frac:.2%}")
                
                try:
                    loss, grad_norm = self.train_step(rgb, nir, mask)
                except tf.errors.InvalidArgumentError as e:
                    print(f"⚠️ Assertion failed at step {step}: {e}")
                    nan_detected = True
                    break
                
                # Early NaN detection (fixed: convert tensor to Python bool)
                if bool(tf.math.is_nan(loss).numpy()) and not nan_detected:
                    print(f"⚠️ NaN detected at step {step}! Stopping epoch.")
                    nan_detected = True
                    break
                    
                if step % 50 == 0:
                    lr = self.lr_schedule(self.optimizer.iterations)
                    print(f"Step {step} | L1: {self.train_l1.result():.4f} | GradNorm: {grad_norm:.2f} | LR: {lr:.2e}")
            
            if nan_detected:
                print("Training stopped due to NaN. Try reducing LR further.")
                break

            # Validation
            for rgb, nir, mask in val_ds:
                self.val_step(rgb, nir, mask)

            train_l1 = self.train_l1.result().numpy()
            val_l1 = self.val_l1.result().numpy()
            val_mrae = self.val_mrae.result().numpy()
            val_sam = self.val_sam.result().numpy()

            print(f"\nEpoch {epoch+1}/{self.config.EPOCHS}")
            print(f"  Train L1: {train_l1:.4f}")
            print(f"  Val L1: {val_l1:.4f} | MRAE: {val_mrae:.4f} | SAM: {val_sam:.4f}")

            if val_l1 < self.best_val:
                self.best_val = val_l1
                self.model.save_weights(os.path.join(self.checkpoint_dir, 'mstpp_best.weights.h5'))
                print("  Best model saved!")

            self.save_checkpoint(epoch + 1)
            print()

print("MSTPPTrainer defined with warmup + cosine decay")

## 6. Training

In [ ]:
# GCS authentication (Colab)
try:
    from google.colab import auth
    auth.authenticate_user()
except Exception:
    pass

# Mount Drive if needed
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

# Clear Keras session to avoid "Variable already initialized" error when resuming
keras.backend.clear_session()

ext = '.tfrecord.gz' if config.TFRECORD_COMPRESSION == 'GZIP' else '.tfrecord'
train_pattern = f"gs://{config.BUCKET}/{config.FOLDER}/train/*{ext}"
test_pattern = f"gs://{config.BUCKET}/{config.FOLDER}/test/*{ext}"

train_ds, val_ds = load_train_val_split(train_pattern, config, val_split=0.1)
test_ds = load_dataset(test_pattern, config, is_training=False, tag='test')

train_files = tf.io.gfile.glob(train_pattern)
estimated_examples = len(train_files) * config.EST_EXAMPLES_PER_FILE
steps_per_epoch = max(1, estimated_examples // config.BATCH_SIZE)

print(f"\nDataset stats:")
print(f"  Estimated examples: {estimated_examples:,}")
print(f"  Steps per epoch: {steps_per_epoch:,}")

trainer = MSTPPTrainer(config, checkpoint_dir='checkpoints_mstpp', steps_per_epoch=steps_per_epoch)
trainer.model.summary()
trainer.train(train_ds, val_ds, resume=True, from_drive=True)

In [ ]:
# ============================================================================
# DIAGNOSTIC: Find where NaN appears in the model
# ============================================================================

def diagnose_nan():
    """Run diagnostic to find where NaN appears in forward/backward pass."""
    print("=" * 60)
    print("NaN DIAGNOSTIC TEST")
    print("=" * 60)
    
    # Get a sample batch
    for rgb, nir, mask in train_ds.take(1):
        print(f"\n1. INPUT DATA:")
        print(f"   RGB shape: {rgb.shape}, range: [{tf.reduce_min(rgb):.4f}, {tf.reduce_max(rgb):.4f}]")
        print(f"   NIR shape: {nir.shape}, range: [{tf.reduce_min(nir):.4f}, {tf.reduce_max(nir):.4f}]")
        print(f"   Has NaN in RGB: {tf.reduce_any(tf.math.is_nan(rgb))}")
        print(f"   Has NaN in NIR: {tf.reduce_any(tf.math.is_nan(nir))}")
        
        # Build a fresh model for testing
        test_model = build_mstpp(config)
        
        # Get the MSTPP layer
        mstpp_layer = test_model.layers[1]
        
        print(f"\n2. FORWARD PASS (layer by layer):")
        
        # Input projection
        x = mstpp_layer.input_proj(rgb)
        print(f"   After input_proj: range [{tf.reduce_min(x):.4f}, {tf.reduce_max(x):.4f}], NaN: {tf.reduce_any(tf.math.is_nan(x))}")
        
        # First SST stage - go deeper
        stage = mstpp_layer.stages[0]
        
        # Encoder level 0
        enc0_out = x
        for i, blk in enumerate(stage.enc0):
            # Inside SAB: norm1 -> attn -> add -> norm2 -> ffn -> add
            normed = blk.norm1(enc0_out)
            print(f"   Stage0.enc0[{i}].norm1: range [{tf.reduce_min(normed):.4f}, {tf.reduce_max(normed):.4f}], NaN: {tf.reduce_any(tf.math.is_nan(normed))}")
            
            # Inside SMSA
            attn_layer = blk.attn
            shape = tf.shape(normed)
            B, H, W, C = shape[0], shape[1], shape[2], shape[3]
            
            qkv = attn_layer.qkv(normed)
            print(f"   Stage0.enc0[{i}].attn.qkv: range [{tf.reduce_min(qkv):.4f}, {tf.reduce_max(qkv):.4f}], NaN: {tf.reduce_any(tf.math.is_nan(qkv))}")
            
            qkv_reshaped = tf.reshape(qkv, [B, H * W, 3, attn_layer.num_heads, attn_layer.head_dim])
            qkv_t = tf.transpose(qkv_reshaped, [2, 0, 3, 4, 1])
            q, k, v = qkv_t[0], qkv_t[1], qkv_t[2]
            
            # Standard scaled attention (no L2 normalization)
            seq_len = tf.cast(H * W, tf.float32)
            scale = 1.0 / seq_len  # Aggressive scaling (matches SMSA)
            
            attn_scores = tf.matmul(q, k, transpose_b=True) * scale
            print(f"   Attention scores: range [{tf.reduce_min(attn_scores):.4f}, {tf.reduce_max(attn_scores):.4f}], NaN: {tf.reduce_any(tf.math.is_nan(attn_scores))}")
            
            attn_clipped = tf.clip_by_value(attn_scores, -30.0, 30.0)
            attn_softmax = tf.nn.softmax(attn_clipped, axis=-1)
            print(f"   Attention softmax: range [{tf.reduce_min(attn_softmax):.4f}, {tf.reduce_max(attn_softmax):.4f}], NaN: {tf.reduce_any(tf.math.is_nan(attn_softmax))}")
            
            # Full block output
            enc0_out = blk(enc0_out)
            print(f"   Stage0.enc0[{i}] output: range [{tf.reduce_min(enc0_out):.4f}, {tf.reduce_max(enc0_out):.4f}], NaN: {tf.reduce_any(tf.math.is_nan(enc0_out))}")
            
            if tf.reduce_any(tf.math.is_nan(enc0_out)):
                print("   ⚠️ NaN detected! Stopping diagnostic.")
                break
        
        print(f"\n3. FULL FORWARD PASS:")
        pred = test_model(rgb, training=True)
        print(f"   Prediction: range [{tf.reduce_min(pred):.4f}, {tf.reduce_max(pred):.4f}], NaN: {tf.reduce_any(tf.math.is_nan(pred))}")
        
        if not tf.reduce_any(tf.math.is_nan(pred)):
            print(f"\n4. BACKWARD PASS (gradient computation):")
            with tf.GradientTape() as tape:
                pred = test_model(rgb, training=True)
                loss = mstpp_loss(nir, pred, mask)
                print(f"   Loss: {loss:.6f}, NaN: {tf.math.is_nan(loss)}")
            
            grads = tape.gradient(loss, test_model.trainable_variables)
            
            # Check gradients
            nan_grad_vars = []
            for var, grad in zip(test_model.trainable_variables, grads):
                if grad is not None:
                    has_nan = tf.reduce_any(tf.math.is_nan(grad))
                    has_inf = tf.reduce_any(tf.math.is_inf(grad))
                    grad_max = tf.reduce_max(tf.abs(grad))
                    if has_nan or has_inf:
                        nan_grad_vars.append((var.name, has_nan.numpy(), has_inf.numpy(), grad_max.numpy()))
            
            if nan_grad_vars:
                print(f"   ⚠️ Found {len(nan_grad_vars)} variables with NaN/Inf gradients:")
                for name, has_nan, has_inf, grad_max in nan_grad_vars[:10]:
                    print(f"      - {name}: NaN={has_nan}, Inf={has_inf}, MaxAbs={grad_max}")
            else:
                print("   ✅ All gradients are finite!")
            
            # Compute global norm
            flat_grads = [tf.reshape(g, [-1]) for g in grads if g is not None]
            all_grads = tf.concat(flat_grads, axis=0)
            global_norm = tf.norm(all_grads)
            print(f"   Global gradient norm: {global_norm:.4f}")
        
        del test_model
        break

# Run diagnostic
diagnose_nan()

## 7. Evaluation (Test Set)

In [ ]:
if test_ds is not None:
    l1_metric = tf.keras.metrics.Mean()
    mrae_metric = tf.keras.metrics.Mean()
    rmse_metric = tf.keras.metrics.Mean()
    sam_metric = tf.keras.metrics.Mean()

    model = build_mstpp(config)
    weights_path = os.path.join('checkpoints_mstpp', 'mstpp_best.weights.h5')
    if os.path.exists(weights_path):
        model.load_weights(weights_path)
        print(f"Loaded weights from {weights_path}")

    for rgb, nir, mask in test_ds:
        pred = model(rgb, training=False)
        l1_metric.update_state(l1_loss_masked(nir, pred, mask))
        mrae_metric.update_state(mrae_metric_masked(nir, pred, mask))
        rmse_metric.update_state(rmse_metric_masked(nir, pred, mask))
        sam_metric.update_state(sam_metric_masked(nir, pred, mask))

    print("\nTest Metrics (MST++):")
    print(f"  L1 (MAE): {l1_metric.result().numpy():.4f}")
    print(f"  MRAE:     {mrae_metric.result().numpy():.4f}")
    print(f"  RMSE:     {rmse_metric.result().numpy():.4f}")
    print(f"  SAM:      {sam_metric.result().numpy():.4f}")

## 8. Visualization

In [ ]:
import matplotlib.pyplot as plt

def visualize_predictions(model, dataset, num_samples=3):
    """Visualize MST++ predictions vs ground truth."""
    for batch in dataset.take(1):
        rgb_batch, nir_batch, mask_batch = batch
        pred_batch = model(rgb_batch, training=False)

        n = min(num_samples, rgb_batch.shape[0])
        fig, axes = plt.subplots(n, 4, figsize=(16, 4*n))
        if n == 1:
            axes = [axes]
        
        for i in range(n):
            axes[i][0].imshow(rgb_batch[i].numpy())
            axes[i][0].set_title('RGB Input')
            axes[i][0].axis('off')
            
            axes[i][1].imshow(nir_batch[i][..., -1].numpy(), cmap='RdYlGn', vmin=0, vmax=1)
            axes[i][1].set_title('GT NIR (B8)')
            axes[i][1].axis('off')
            
            axes[i][2].imshow(pred_batch[i][..., -1].numpy(), cmap='RdYlGn', vmin=0, vmax=1)
            axes[i][2].set_title('Pred NIR (B8)')
            axes[i][2].axis('off')
            
            error = tf.abs(nir_batch[i][..., -1] - pred_batch[i][..., -1]).numpy()
            im = axes[i][3].imshow(error, cmap='hot', vmin=0, vmax=0.2)
            axes[i][3].set_title('Absolute Error')
            axes[i][3].axis('off')
            plt.colorbar(im, ax=axes[i][3], fraction=0.046)
        
        plt.tight_layout()
        plt.savefig('mstpp_predictions.png', dpi=150, bbox_inches='tight')
        plt.show()
        print("Saved to mstpp_predictions.png")

# Example usage:
model = build_mstpp(config)
# model.load_weights('checkpoints_mstpp/mstpp_best.weights.h5')
visualize_predictions(model, test_ds)